In [1]:
import pandas as pd

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/karlaazuniga/etl-data-pipeline2516662022/refs/heads/main/data/raw/A_cursos.csv"

df = pd.read_csv(url)

df.head()

,id_curso,curso,docente,creditos
0,C200,Matemática,Lic. Hernández,3
1,C201,Programación,Mtra. Pérez,4
2,C202,Base de Datos,Mtra. Rivera,4
3,C203,Redacción,Ing. López,4
4,C204,Inglés,Ing. Romero,5


In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_curso  13 non-null     object
 1   curso     13 non-null     object
 2   docente   13 non-null     object
 3   creditos  13 non-null     object
dtypes: object(4)
memory usage: 548.0+ bytes


,0
id_curso,0
curso,0
docente,0
creditos,0


In [4]:
cursos = df.copy()

# limpiar nombres de columnas
cursos.columns = cursos.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in cursos.select_dtypes(include='object').columns:
    cursos[col] = cursos[col].astype(str).str.strip()

# convertir vacíos en null
cursos = cursos.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
cursos = cursos.drop_duplicates()

In [5]:
cursos['creditos'] = pd.to_numeric(
    cursos['creditos'],
    errors='coerce'
)

In [6]:
cursos['curso'] = cursos['curso'].astype(str).str.strip().str.title()
cursos['docente'] = cursos['docente'].astype(str).str.strip().str.title()

In [7]:
validos = cursos[
    cursos['creditos'].notna() &
    cursos['docente'].notna() &
    cursos['curso'].notna()
].copy()

rechazados = cursos[
    cursos['creditos'].isna() |
    cursos['docente'].isna() |
    cursos['curso'].isna()
].copy()

In [8]:
def motivo(row):
    motivos = []

    if pd.isna(row['creditos']):
        motivos.append("creditos_invalido")

    if pd.isna(row['docente']):
        motivos.append("docente_invalido")

    if pd.isna(row['curso']):
        motivos.append("curso_invalido")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [9]:
validos.to_csv("cursos_curated.csv", index=False)
rechazados.to_csv("cursos_rejects.csv", index=False)